In [1]:
import os
import sys
import json
import copy
import random
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score, classification_report

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AdamW,
    get_linear_schedule_with_warmup
)

# Import the feature pipeline utilities (for data loading and propagation)
from Rumors_Classifier.feature_pipeline import (
    load_tweets,
    load_propagation_counts,
    ArabicTextPreprocessor
)

In [2]:
import logging
import os
from datetime import datetime


def setup_logger(name="training", log_dir="logs"):
    os.makedirs(log_dir, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = os.path.join(log_dir, f"training_{timestamp}.log")

    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)

    # Prevent duplicate handlers
    if logger.handlers:
        return logger

    formatter = logging.Formatter(
        fmt="%(asctime)s | %(levelname)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S"
    )

    # File handler
    file_handler = logging.FileHandler(log_file)
    file_handler.setFormatter(formatter)

    # Console handler
    console_handler = logging.StreamHandler()
    console_handler.setFormatter(formatter)

    logger.addHandler(file_handler)
    logger.addHandler(console_handler)

    return logger

In [3]:
# Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Paths to data (adjust to your local setup)
DATA_ROOT = Path("../../ArCOV19-Rumors/tweet_verification")
TWEETS_PATH = DATA_ROOT / "Tweets.txt"
REPLIES_PATH = DATA_ROOT / "propagation_networks/replies"
RETWEETS_PATH = DATA_ROOT / "propagation_networks/retweets"


Using device: cuda


In [ ]:
# Load raw tweets (tab-separated)
tweets_df = load_tweets(TWEETS_PATH)

# Add propagation features
reply_counts = load_propagation_counts(REPLIES_PATH)
retweet_counts = load_propagation_counts(RETWEETS_PATH)
tweets_df['num_replies'] = tweets_df['tweetID'].map(reply_counts).fillna(0).astype(int)
tweets_df['num_retweets'] = tweets_df['tweetID'].map(retweet_counts).fillna(0).astype(int)

# Keep only needed columns: tweetText (raw), label (bool), and propagation counts
df = tweets_df[['tweetText', 'label', 'num_replies', 'num_retweets']].copy()

# Minimal preprocessing: remove extra whitespace only
def minimal_preprocess(text):
    return ' '.join(text.split())

df['text'] = df['tweetText'].apply(minimal_preprocess)

# Convert label to int (0/1)
df['label'] = df['label'].astype(int)

print(f"Dataset shape: {df.shape}")
print(f"Label distribution:\n{df['label'].value_counts()}")
print(df.head())

In [5]:
class MARBERTClassifier:
    def __init__(self, model_name='UBC-NLP/MARBERT', num_labels=2, max_seq_len=128):
        self.model_name = model_name
        self.num_labels = num_labels
        self.max_seq_len = max_seq_len
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
        self.model.to(device)

    def tokenize(self, texts, max_len=None):
        if max_len is None:
            max_len = self.max_seq_len
        return self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=max_len,
            return_tensors='pt'
        )

    def forward(self, input_ids, attention_mask):
        return self.model(input_ids=input_ids, attention_mask=attention_mask)

In [6]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

In [7]:
class MARBERTFusionClassifier(nn.Module):
    """MARBERT + propagation features (replies & retweets)"""
    def __init__(self, model_name, num_labels, num_prop_features=2):
        super().__init__()
        self.bert = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
        # Replace classifier: we will use the pooled output
        self.bert.classifier = nn.Identity()
        self.hidden_size = self.bert.config.hidden_size
        self.num_prop_features = num_prop_features
        self.fusion_layer = nn.Linear(self.hidden_size + num_prop_features, num_labels)

    def forward(self, input_ids, attention_mask, prop_features):
        outputs = self.bert.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.pooler_output  # (batch, hidden_size)
        combined = torch.cat([pooled, prop_features], dim=1)
        logits = self.fusion_layer(combined)
        return logits

In [8]:
class TweetDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        encoding = self.tokenizer(
            row['text'],
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(row['label'], dtype=torch.long)
        }

class FusionDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        encoding = self.tokenizer(
            row['text'],
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt'
        )
        # log-transform propagation counts to reduce skew
        replies = np.log1p(row['num_replies'])
        retweets = np.log1p(row['num_retweets'])
        prop = torch.tensor([replies, retweets], dtype=torch.float)
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'prop_features': prop,
            'label': torch.tensor(row['label'], dtype=torch.long)
        }

In [9]:
def train_epoch(model, dataloader, optimizer, scheduler, loss_fn, logger, fusion=False):
    model.train()
    total_loss = 0
    running_loss = 0
    all_preds, all_labels = [], []

    for batch_idx, batch in enumerate(tqdm(dataloader, desc='Training')):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        if fusion:
            prop_features = batch['prop_features'].to(device)
            logits = model(input_ids, attention_mask, prop_features)
        else:
            logits = model(input_ids, attention_mask).logits

        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()

        running_loss += loss.item()
        if (batch_idx + 1) % 100 == 0:
            current_lr = scheduler.get_last_lr()[0]
            logger.info(
                f"Batch {batch_idx+1}/{len(dataloader)} | "
                f"Loss: {running_loss/100:.4f} | LR: {current_lr:.8f}"
            )
            running_loss = 0

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    f1 = f1_score(all_labels, all_preds, average='macro')
    return avg_loss, f1

def evaluate(model, dataloader, loss_fn, fusion=False, return_preds=False):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Evaluating'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            if fusion:
                prop_features = batch['prop_features'].to(device)
                logits = model(input_ids, attention_mask, prop_features)
            else:
                logits = model(input_ids, attention_mask).logits

            loss = loss_fn(logits, labels)
            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    f1 = f1_score(all_labels, all_preds, average='macro')
    if return_preds:
        return avg_loss, f1, all_labels, all_preds

    return avg_loss, f1, None, None

In [10]:
def run_experiment(df, config, folds=None, verbose=True):
    """
    config: dict with keys:
        model_name, num_labels, max_seq_len, batch_size, epochs,
        lr_backbone, lr_head, loss_type ('focal' or 'ce'),
        early_stop_metric ('f1' or 'loss'), patience,
        use_propagation (bool)
    """

    logger = setup_logger()

    # Extract parameters
    model_name = config['model_name']
    num_labels = config['num_labels']
    max_seq_len = config['max_seq_len']
    batch_size = config['batch_size']
    epochs = config['epochs']
    lr_backbone = config['lr_backbone']
    lr_head = config['lr_head']
    loss_type = config['loss_type']
    early_stop_metric = config['early_stop_metric']
    patience = config['patience']
    use_propagation = config.get('use_propagation', False)

    labels = None
    preds = None

    logger.info("=" * 80)
    logger.info("Starting experiment")
    logger.info("=" * 80)

    for key, value in config.items():
        logger.info(f"{key}: {value}")

    # Prepare folds if not given
    if folds is None:
        skf = StratifiedKFold(n_splits=1, shuffle=True, random_state=42)
        folds = list(skf.split(df, df['label']))

    val_f1_scores = []

    for fold, (train_idx, val_idx) in enumerate(folds):
        if verbose:
            logger.info(
                f"{'='*60}\n"
                f"Fold {fold+1}/{len(folds)}\n"
                f"Train samples: {len(train_idx)} | "
                f"Validation samples: {len(val_idx)}"
            )
        train_df = df.iloc[train_idx].reset_index(drop=True)
        val_df = df.iloc[val_idx].reset_index(drop=True)

        # Initialize model
        if use_propagation:
            model = MARBERTFusionClassifier(model_name, num_labels, num_prop_features=2)
            model.to(device)
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            dataset_class = FusionDataset
        else:
            classifier = MARBERTClassifier(model_name, num_labels, max_seq_len)
            model = classifier.model
            tokenizer = classifier.tokenizer
            dataset_class = TweetDataset

        total_params = sum(p.numel() for p in model.parameters())
        trainable_params = sum(
            p.numel() for p in model.parameters() if p.requires_grad
        )

        logger.info(
            f"Model initialized: {model.__class__.__name__}"
        )

        logger.info(
            f"Total parameters: {total_params:,}"
        )

        logger.info(
            f"Trainable parameters: {trainable_params:,}"
        )

        # DataLoaders
        train_dataset = dataset_class(train_df, tokenizer, max_seq_len)
        val_dataset = dataset_class(val_df, tokenizer, max_seq_len)
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

        # Differential LR: identify head parameters
        if use_propagation:
            head_params = [p for n, p in model.named_parameters() if 'fusion_layer' in n]
            backbone_params = [p for n, p in model.named_parameters() if 'fusion_layer' not in n]
        else:
            head_params = [p for n, p in model.named_parameters() if 'classifier' in n or 'score' in n]
            backbone_params = [p for n, p in model.named_parameters() if not ('classifier' in n or 'score' in n)]

        optimizer_grouped = [
            {'params': backbone_params, 'lr': lr_backbone},
            {'params': head_params, 'lr': lr_head}
        ]
        optimizer = AdamW(optimizer_grouped, weight_decay=0.01)
        total_steps = len(train_loader) * epochs
        scheduler = get_linear_schedule_with_warmup(
            optimizer, num_warmup_steps=0.1*total_steps, num_training_steps=total_steps
        )

        # Loss function
        if loss_type == 'focal':
            loss_fn = FocalLoss(gamma=2.0, alpha=None)
        else:  # weighted CE
            class_counts = train_df['label'].value_counts().sort_index().values
            class_weights = 1.0 / class_counts
            class_weights = class_weights / class_weights.sum() * len(class_counts)
            alpha = torch.tensor(class_weights, dtype=torch.float).to(device)
            loss_fn = nn.CrossEntropyLoss(weight=alpha)

        # Training loop with early stopping
        best_score = -np.inf if early_stop_metric == 'f1' else np.inf
        best_model_state = None
        patience_counter = 0

        for epoch in range(epochs):
            if verbose:
                logger.info(f"Epoch {epoch+1}/{epochs}")
            train_loss, train_f1 = train_epoch(
                model, train_loader, optimizer, scheduler, loss_fn, logger, fusion=use_propagation
            )
            val_loss, val_f1, _, _ = evaluate(
                model, val_loader, loss_fn, fusion=use_propagation
            )
            if verbose:
                logger.info(
                    f"Epoch {epoch+1}/{epochs} Results | "
                    f"Train Loss={train_loss:.4f} | "
                    f"Train F1={train_f1:.4f} | "
                    f"Val Loss={val_loss:.4f} | "
                    f"Val F1={val_f1:.4f}"
                )

            # Update best
            if early_stop_metric == 'f1':
                if val_f1 > best_score:
                    best_score = val_f1
                    best_model_state = copy.deepcopy(model.state_dict())
                    patience_counter = 0

                    logger.info(
                        f"New best validation F1: {val_f1:.4f}"
                    )

                else:
                    patience_counter += 1

            else:
                if val_loss < best_score:
                    best_score = val_loss
                    best_model_state = copy.deepcopy(model.state_dict())
                    patience_counter = 0

                    logger.info(
                        f"New best validation loss: {val_loss:.4f}"
                    )

                else:
                    patience_counter += 1

            if patience_counter >= patience:
                if verbose:
                    logger.warning(
                        f"Early stopping triggered at epoch {epoch+1}. "
                        f"No improvement for {patience} epochs."
                    )
                break

        # Restore best model
        if best_model_state is not None:
            model.load_state_dict(best_model_state)
        # Final evaluation on val set
        _, best_val_f1, labels, preds = evaluate(model, val_loader, loss_fn, fusion=use_propagation, return_preds=True)
        val_f1_scores.append(best_val_f1)

    avg_f1 = np.mean(val_f1_scores)
    std_f1 = np.std(val_f1_scores)
    if verbose:
        logger.info("=" * 80)
        logger.info(
            f"Experiment completed | "
            f"Mean F1: {avg_f1:.4f} ± {std_f1:.4f}"
        )
        logger.info("=" * 80)

        if labels is not None and preds is not None:
            report = classification_report(
                labels,
                preds,
                digits=4
            )

            logger.info(
                "\nClassification Report:\n" + report
            )
    return avg_f1, std_f1

In [11]:
Differential_CONFIG = {
    'model_name': 'UBC-NLP/MARBERT',
    'num_labels': 2,
    'max_seq_len': 128,
    'batch_size': 32,
    'epochs': 25,
    'lr_backbone': 2e-5,
    'lr_head': 5e-5,
    'loss_type': 'focal',
    'early_stop_metric': 'f1',
    'patience': 2,
    'use_propagation': False
}

In [12]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
five_folds = list(skf.split(df, df['label']))
skf_three = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
three_folds = list(skf_three.split(df, df['label']))

In [17]:
# Baseline
config_diff = Differential_CONFIG.copy()
config_diff['lr_backbone'] = 2e-5
config_diff['lr_head'] = 5e-5

# Counterpart
config_same = Differential_CONFIG.copy()
config_same['lr_backbone'] = 5e-5
config_same['lr_head'] = 5e-5

print("=== Differential LR ===")
avg_f1_diff, std_f1_diff = run_experiment(df, config_diff, folds=five_folds, verbose=False)
print(f"Result: {avg_f1_diff:.4f} ± {std_f1_diff:.4f}")

print("\n=== Same LR ===")
avg_f1_same, std_f1_same = run_experiment(df, config_same, folds=five_folds, verbose=False)
print(f"Result: {avg_f1_same:.4f} ± {std_f1_same:.4f}")

print(f"\nImprovement: {avg_f1_diff - avg_f1_same:.4f} F1")

=== Differential LR (baseline) ===


tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Evaluating:   0%|          | 0/23 [00:00<?, ?it/s]

Training:   0%|          | 0/90 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/23 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/23 [00:00<?, ?it/s]

Result: 0.9634 ± 0.0081

Improvement: 0.0017 F1


In [18]:
config_focal = Differential_CONFIG.copy()
config_focal['loss_type'] = 'focal'

config_ce = Differential_CONFIG.copy()
config_ce['loss_type'] = 'ce'

print("=== Focal Loss ===")
avg_f1_focal, std_f1_focal = run_experiment(df, config_focal, folds=five_folds, verbose=False)
print(f"Result: {avg_f1_focal:.4f} ± {std_f1_focal:.4f}")

print("\n=== Weighted CE ===")
avg_f1_ce, std_f1_ce = run_experiment(df, config_ce, folds=five_folds, verbose=False)
print(f"Result: {avg_f1_ce:.4f} ± {std_f1_ce:.4f}")

print(f"\nImprovement: {avg_f1_focal - avg_f1_ce:.4f} F1")

=== Focal Loss (baseline) ===


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at UBC-NLP/MARBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\LENOVO\Personal Projects\NLP\arabic_rumor_scanner\ARS_venv\Lib\site-packages\transformers\optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Training:   0%|          | 0/90 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/23 [00:00<?, ?it/s]

Training:   0%|          | 0/90 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/23 [00:00<?, ?it/s]

Training:   0%|          | 0/90 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/23 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/23 [00:00<?, ?it/s]

Result: 0.9623 ± 0.0098

Improvement: -0.0003 F1


In [19]:
config_128 = Differential_CONFIG.copy()
config_128['max_seq_len'] = 128

config_512 = Differential_CONFIG.copy()
config_512['max_seq_len'] = 512
config_512['batch_size'] = 8

print("=== max_seq_len=128 ===")
avg_f1_128, std_f1_128 = run_experiment(df, config_128, folds=five_folds, verbose=False)
print(f"Result: {avg_f1_128:.4f} ± {std_f1_128:.4f}")

print("\n=== max_seq_len=512 ===")
avg_f1_512, std_f1_512 = run_experiment(df, config_512, folds=five_folds, verbose=False)
print(f"Result: {avg_f1_512:.4f} ± {std_f1_512:.4f}")

print(f"\nImprovement: {avg_f1_128 - avg_f1_512:.4f} F1")

=== max_seq_len=128 (baseline) ===


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at UBC-NLP/MARBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\LENOVO\Personal Projects\NLP\arabic_rumor_scanner\ARS_venv\Lib\site-packages\transformers\optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Training:   0%|          | 0/90 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/23 [00:00<?, ?it/s]

Training:   0%|          | 0/90 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/90 [00:00<?, ?it/s]

Training:   0%|          | 0/359 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/90 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/90 [00:00<?, ?it/s]

Result: 0.9617 ± 0.0064

Improvement: 0.0011 F1


In [20]:
config_f1_es = Differential_CONFIG.copy()
config_f1_es['early_stop_metric'] = 'f1'

config_loss_es = Differential_CONFIG.copy()
config_loss_es['early_stop_metric'] = 'loss'

print("=== ES on F1 ===")
avg_f1_f1es, std_f1_f1es = run_experiment(df, config_f1_es, folds=five_folds, verbose=False)
print(f"Result: {avg_f1_f1es:.4f} ± {std_f1_f1es:.4f}")

print("\n=== ES on loss ===")
avg_f1_losses, std_f1_losses = run_experiment(df, config_loss_es, folds=five_folds, verbose=False)
print(f"Result: {avg_f1_losses:.4f} ± {std_f1_losses:.4f}")

print(f"\nImprovement: {avg_f1_f1es - avg_f1_losses:.4f} F1")

=== ES on F1 (baseline) ===


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at UBC-NLP/MARBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\LENOVO\Personal Projects\NLP\arabic_rumor_scanner\ARS_venv\Lib\site-packages\transformers\optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Training:   0%|          | 0/90 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/23 [00:00<?, ?it/s]

Training:   0%|          | 0/90 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/23 [00:00<?, ?it/s]

Training:   0%|          | 0/90 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/23 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/23 [00:00<?, ?it/s]

Result: 0.9573 ± 0.0066

Improvement: 0.0092 F1


In [23]:
config_five = Differential_CONFIG.copy()
config_three = Differential_CONFIG.copy()

print("=== Five Folds ===")
avg_f1_five, std_f1_five = run_experiment(df, config_five, folds=five_folds, verbose=False)
print(f"Result: {avg_f1_five:.4f} ± {std_f1_five:.4f}")

print("\n=== Three Folds ===")
avg_f1_three, std_f1_three = run_experiment(df, config_three, folds=three_folds, verbose=False)
print(f"Result: {avg_f1_three:.4f} ± {std_f1_three:.4f}")

print(f"\nImprovement: {avg_f1_five - avg_f1_three:.4f} F1")

=== Five Folds ===


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at UBC-NLP/MARBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\LENOVO\Personal Projects\NLP\arabic_rumor_scanner\ARS_venv\Lib\site-packages\transformers\optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Training:   0%|          | 0/90 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/23 [00:00<?, ?it/s]

Training:   0%|          | 0/90 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/38 [00:00<?, ?it/s]

Training:   0%|          | 0/75 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/38 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/38 [00:00<?, ?it/s]

Result: 0.9634 ± 0.0052

Improvement: -0.0014 F1


In [13]:
config_text = Differential_CONFIG.copy()
config_text['use_propagation'] = False

config_fusion = Differential_CONFIG.copy()
config_fusion['use_propagation'] = True
# fusion model may need slightly different LR; we keep the same.

print("=== Text-only ===")
avg_f1_text, std_f1_text = run_experiment(df, config_text, folds=three_folds, verbose=True)
print(f"Result: {avg_f1_text:.4f} ± {std_f1_text:.4f}")

print("\n=== Hybrid fusion ===")
avg_f1_fusion, std_f1_fusion = run_experiment(df, config_fusion, folds=three_folds, verbose=True)
print(f"Result: {avg_f1_fusion:.4f} ± {std_f1_fusion:.4f}")

print(f"\nImprovement: {avg_f1_fusion - avg_f1_text:.4f} F1")

2026-07-07 05:29:09 | INFO | ================================================================================
2026-07-07 05:29:09 | INFO | Starting experiment
2026-07-07 05:29:09 | INFO | ================================================================================
2026-07-07 05:29:09 | INFO | model_name: UBC-NLP/MARBERT
2026-07-07 05:29:09 | INFO | num_labels: 2
2026-07-07 05:29:09 | INFO | max_seq_len: 128
2026-07-07 05:29:09 | INFO | batch_size: 32
2026-07-07 05:29:09 | INFO | epochs: 25
2026-07-07 05:29:09 | INFO | lr_backbone: 2e-05
2026-07-07 05:29:09 | INFO | lr_head: 5e-05
2026-07-07 05:29:09 | INFO | loss_type: focal
2026-07-07 05:29:09 | INFO | early_stop_metric: f1
2026-07-07 05:29:09 | INFO | patience: 2
2026-07-07 05:29:09 | INFO | use_propagation: False
2026-07-07 05:29:09 | INFO | ============================================================
Fold 1/3
Train samples: 2389 | Validation samples: 1195


=== Text-only ===


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at UBC-NLP/MARBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
2026-07-07 05:29:12 | INFO | Model initialized: BertForSequenceClassification
2026-07-07 05:29:12 | INFO | Total parameters: 162,842,882
2026-07-07 05:29:12 | INFO | Trainable parameters: 162,842,882
C:\Users\LENOVO\Personal Projects\NLP\arabic_rumor_scanner\ARS_venv\Lib\site-packages\transformers\optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
2026-07-07 05:29:12 | INFO | Epoch 1/25


Training:   0%|          | 0/75 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/38 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/38 [00:00<?, ?it/s]

2026-07-07 06:42:54 | INFO | Epoch 9/25 Results | Train Loss=0.0001 | Train F1=1.0000 | Val Loss=0.0783 | Val F1=0.9572
2026-07-07 06:42:54 | WARNING | Early stopping triggered at epoch 9. No improvement for 2 epochs.


Evaluating:   0%|          | 0/38 [00:00<?, ?it/s]

C:\Users\LENOVO\Personal Projects\NLP\arabic_rumor_scanner\ARS_venv\Lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
C:\Users\LENOVO\Personal Projects\NLP\arabic_rumor_scanner\ARS_venv\Lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
C:\Users\LENOVO\Personal Projects\NLP\arabic_rumor_scanner\ARS_venv\Lib\site-packages\numpy\core\_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
C:\Users\LENOVO\Personal Projects\NLP\arabic_rumor_scanner\ARS_venv\Lib\site-packages\numpy\core\_methods.py:163: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
C:\Users\LENOVO\Personal Projects\NLP\arabic_rumor_scanner\ARS_venv\Lib\site-packages\numpy\core\_methods.py:198: RuntimeWarning: invalid va

Result: nan ± nan

Improvement: nan F1


- Focal loss vs. CE: No major performance gap between the two. We're keeping focal loss as a precaution in case we hit imbalanced data down the line, but we did note that the current dataset is clean and well-labeled, so it's not causing any bias right now.

- 5 folds vs. 3 folds: We went with 5 initially to match the original paper, but the improvement over 3 folds was minimal—well within the error variance range. So we're dropping to 3 folds to save on training costs, though I'll admit that might change with larger datasets.

- Diff lr vs. same lr: Again, no major differences in our tests. Still, we're sticking with diff lr—it's more of a safety net against unwanted backbone drift over longer runs or bigger datasets, plus it's the industry standard, so it feels right to keep.

- 128 max_len_seq vs. 512: Matched the hypothesis—no meaningful difference. Most tweets are character-limited anyway (free vs. premium accounts), so the text rarely exceeds 128 tokens. On top of that, 128 gives us a massive cost reduction, so it's a no-brainer to settle there.

- Fusion dataset vs. normal dataset: This one actually gave us a slight bump. Adding num_replies and num_retweets improved things just enough that we're keeping the fusion version moving forward.